# Phase 14 — train the small encoderRuns on Colab's free GPU. Set the runtime to **T4 GPU** first:Runtime → Change runtime type → T4 GPU.Training a 22M model on 50,000 examples takes a few minutes there. The same run on alaptop CPU takes hours, which is why this notebook exists.Upload `working.jsonl` from `research/data/` when the last cell asks for it. That isthe hand-labelled test set, and it is how the trained model gets a number.

In [ ]:
!pip -q install sentence-transformersimport nltkfor package in ["wordnet", "omw-1.4", "semcor"]:    nltk.download(package, quiet=True)import torchprint("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")

## The training dataSemCor is 37,000 sentences where a person marked which sense each content wordcarries. Each example comes out shaped like the question asked at run time: asentence, a target word, the senses it could carry, and which one is right.Words with a single sense are dropped — there is nothing to learn from a choice of one.

In [ ]:
import json, random, collectionsfrom nltk.corpus import semcorfrom nltk.corpus import wordnet as wnMIN_SENSES, MIN_WORDS = 2, 4def examples():    for sentence in semcor.tagged_sents(tag="sem"):        words, tagged = [], []        for chunk in sentence:            leaves = chunk.leaves() if hasattr(chunk, "leaves") else list(chunk)            start = len(words)            words.extend(leaves)            label = getattr(chunk, "label", lambda: None)()            if label is not None and hasattr(label, "synset"):                tagged.append((start, len(leaves), label))        if len(words) < MIN_WORDS:            continue        text = " ".join(words)        for start, length, label in tagged:            try:                synset, lemma = label.synset(), label.name()            except Exception:                continue            if not lemma or synset.pos() not in "nvar":                continue            candidates = [s for s in wn.synsets(lemma, synset.pos())                          if any(l.name() == lemma for l in s.lemmas())]            if len(candidates) < MIN_SENSES or synset not in candidates:                continue            yield {"text": text, "word": " ".join(words[start:start + length]),                   "lemma": lemma.replace("_", " ").lower(), "pos": synset.pos(),                   "key": synset.name(),                   "candidates": [s.name() for s in candidates]}rows = list(examples())random.Random(17).shuffle(rows)first = sum(1 for r in rows if r["candidates"][0] == r["key"])print(f"examples {len(rows):,}")print(f"words    {len({r['lemma'] for r in rows}):,} distinct")print(f"first    {100 * first / len(rows):.1f}% of them are the commonest sense")

## How a line and a sense are written downPhase 12 measured this: a sense reads better as its synonyms, its definition and itsexamples together than as the definition alone — seven points better. WordNet'sdefinitions are abstract, its examples are how people speak, and the question is aline somebody spoke.

In [ ]:
def sense_text(key):    s = wn.synset(key)    return " ".join([", ".join(l.name().replace("_", " ") for l in s.lemmas()) + ":",                     s.definition(), *s.examples()[:2]])def line_text(row):    return f'{row["lemma"]}: {row["text"]}'r = rows[0]print(line_text(r))print(" right:", sense_text(r["key"])[:90])

## Splitting the dataThe split is **by word, not by row**. If `safe` appeared in both halves the modelcould recognise the word rather than read the sentence, and the validation numberwould flatter it.

In [ ]:
rng = random.Random(17)words = sorted({r["lemma"] for r in rows})rng.shuffle(words)held = set(words[: len(words) // 10])train = [r for r in rows if r["lemma"] not in held][:50_000]valid = [r for r in rows if r["lemma"] in held][:2_000]print(f"train {len(train):,} examples, {len({r['lemma'] for r in train}):,} words")print(f"valid {len(valid):,} examples, {len({r['lemma'] for r in valid}):,} words")

## TrainingThree words to know.A **batch** is how many examples the model sees before adjusting itself. Bigger issteadier and needs more memory.An **epoch** is one pass over the training set. More epochs means more chances tolearn and, past a point, more chances to memorise.The **loss** is how wrong the model currently is. Training is making it go down.Watching it on the training data alone tells you nothing — a memorising model shows afalling training loss and a rising validation loss, and that gap is the thing to watch.The wrong answers handed to the model are **other senses of the same word**, notrandom sentences. Telling `safe` the strongbox from `safe` the contraceptive is thejob; telling it from "the weather is nice" is not.

In [ ]:
from sentence_transformers import (InputExample, SentenceTransformer, losses,                                   evaluation)from torch.utils.data import DataLoaderMODEL = "sentence-transformers/all-MiniLM-L6-v2"   # 22M, the one that fits a browserEPOCHS, BATCH = 1, 64def pairs(source):    made = []    for row in source:        wrong = [k for k in row["candidates"] if k != row["key"]]        if wrong:            made.append(InputExample(texts=[line_text(row), sense_text(row["key"]),                                            sense_text(rng.choice(wrong))]))    return mademodel = SentenceTransformer(MODEL)loader = DataLoader(pairs(train), shuffle=True, batch_size=BATCH)loss = losses.MultipleNegativesRankingLoss(model)checker = evaluation.TripletEvaluator.from_input_examples(pairs(valid),                                                          name="held-out words")print("before training:", checker(model))model.fit(train_objectives=[(loader, loss)], evaluator=checker, epochs=EPOCHS,          warmup_steps=int(0.1 * len(loader)), output_path="model",          show_progress_bar=True)print("after training:", checker(model))

## The number that countsEverything above is measured on SemCor, which is books and journalism. The test set issubtitles, hand-labelled, and never seen by the model. Upload `working.jsonl` now.

In [ ]:
from google.colab import filesuploaded = files.upload()   # choose research/data/working.jsonl

In [ ]:
from sentence_transformers import utiltest = [json.loads(l) for l in open("working.jsonl", encoding="utf-8")]def score(encoder):    lines = encoder.encode([line_text(r) for r in test], convert_to_tensor=True,                           normalize_embeddings=True)    at = collections.Counter()    for row, line in zip(test, lines):        texts = [sense_text(s["key"]) for s in row["senses"]]        senses = encoder.encode(texts, convert_to_tensor=True,                                normalize_embeddings=True)        order = util.cos_sim(line, senses)[0].argsort(descending=True).tolist()        keys = [row["senses"][i]["key"] for i in order]        for depth in (1, 3, 5):            if set(keys[:depth]) & set(row["label"]):                at[depth] += 1    return {d: 100 * at[d] / len(test) for d in (1, 3, 5)}base = sum(1 for r in test if r["senses"][0]["key"] in r["label"])print(f"first sense in the dictionary   {100 * base / len(test):.1f}%")print(f"untrained 22M                   {score(SentenceTransformer(MODEL))}")print(f"trained   22M                   {score(model)}")

In [ ]:
!zip -qr model.zip modelfrom google.colab import filesfiles.download("model.zip")